In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

In [ ]:
train_df = train_identity.merge(train_transaction)

In [ ]:
train_transaction.head()

In [ ]:
train_transaction.shape

In [ ]:
train_transaction.isna().sum()

### Dropping v339

In [ ]:
for i in train_transaction.columns:
    no = train_transaction[i].isna().sum()
    if no > 50000:
        print(i,':',no.sum())
print(f'Highest missing val ;{i}')
    

In [ ]:
small_df = train_transaction.dropna(subset=['V339'])

In [ ]:
small_df.head()

In [ ]:
small_df.shape

In [ ]:
small_df.isna().sum()

In [ ]:
for col in small_df.columns:
    num = small_df[col].isna().sum()
    if num > 0:
        print(col,':',num.sum())

In [ ]:
cols_to_drop = [col for col in small_df.columns if small_df[col].isna().sum() > 15000]
df_small = small_df.drop(columns=cols_to_drop)
df_small.shape

In [ ]:
for i in df_small.columns:
    print(i, df_small[i].isna().sum())
    

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.countplot(x=df_small['isFraud'])
plt.title('Fraud vs Non-fraud balance comparison')
plt.show()

As shown above there is a class imbalance as fraud does not occur as frequently.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
y = df_small['isFraud']
x = df_small.drop('isFraud',axis=1)

In [ ]:
num_cols = x.select_dtypes(include=['float64','int64']).columns.tolist()
cat_cols = x.select_dtypes(include='object').columns.tolist()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,shuffle=False)

In [ ]:
x_test[num_cols] = num_imputed.transform(x_test[num_cols])
x_test[cat_cols] = cat_imputed.transform(x_test[cat_cols])

In [ ]:
num_imputed = SimpleImputer(strategy='median')
x_train[num_cols]= num_imputed.fit_transform(x_train[num_cols])
cat_imputed = SimpleImputer(strategy='most_frequent')
x_train[cat_cols]=cat_imputed.fit_transform(x_train[cat_cols])

In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

In [ ]:
label_dict = {}
for col in cat_cols:
    label = LabelEncoder()
    label.fit(pd.concat([x_train[col], x_test[col]]).astype(str))
    x_train[col] = label.transform(x_train[col].astype(str))
    x_test[col] = label.transform(x_test[col].astype(str))
    label_dict[col] = label

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
model = LGBMClassifier(n_estimators=500,learning_rate=0.05,num_leaves=31,random_state=42,n_jobs=-1,scale_pos_weight=22 )
model.fit(x_train,y_train)
y_pred = model.predict_proba(x_test)[:, 1] 
roc_auc = roc_auc_score(y_test,y_pred)
avg_prec = average_precision_score(y_test,y_pred)
print(roc_auc)
print(avg_prec)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
y_pred_labels = model.predict(x_test)
cm = confusion_matrix(y_test, y_pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Fraud', 'Fraud'])
disp.plot(cmap='Blues')

In [1]:
from XGBoost import XGBClassifier as xgb

ModuleNotFoundError: No module named 'XGBoost'